# Experimento final · BiRNN con dropout y cabeza no lineal

Misma configuración que el candidato final (`class_weight_power = 0.25`, weight decay 0.1 en puntuación), con
tres cambios en la BiRNN:

| | Candidato final | Este experimento | Por qué |
|---|---|---|---|
| Dropout | 0 | **0.3** | La BiRNN de capitalización sobreajustaba: train 0.805 vs test 0.667 en la época 10. Es el mismo dropout que la red conjunta. |
| Cabeza | `Linear 256 → clases` | **`Linear 256 → 128 → GELU → Dropout → Linear 128 → clases`** | La única no linealidad por token era la tanh de la RNN. Es la misma forma de cabeza que `JointHeads` con capa oculta. |
| Épocas | 10 | **15** | Con dropout puede entrenar más antes de sobreajustar; la de puntuación seguía mejorando en la época 10. |

La cabeza está en `src/problem/heads.py` y se activa con `bidirectional_rnn_head_dim`. Con `None` (el default del
repo) la BiRNN queda igual que antes, y `src/inference.py` deduce la cabeza de los pesos guardados, así que los
`.pkl` viejos siguen cargando.

Tiempo estimado: ~3 minutos por época y por problema, ~1,5 horas.

**En Colab**: Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU**. Se puede correr en paralelo con el otro notebook del experimento final (cada uno entrena un modelo); los dos guardan en `MyDrive/tp2-models` y cada uno reconoce su corrida por el experimento registrado en su `metrics.json`. Con Colab gratis puede que solo haya una GPU disponible a la vez.

## 1. Setup

In [ ]:
import json
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BRANCH = "feat/birnn-cabeza-no-lineal"  # rama con la cabeza no lineal de la BiRNN (src/problem/heads.py)
REPO = "/content/TP2-Aprendizaje-Automatico-UBA"

if IN_COLAB:
    # Los resultados van a Drive: la máquina de Colab se borra al cerrar la sesión
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path(REPO).exists():
        !git clone https://github.com/SebasAndres/TP2-Aprendizaje-Automatico-UBA.git {REPO}
    %cd {REPO}
    !git fetch origin && git checkout {BRANCH} && git pull
    !pip install -q uv && uv sync
    !mkdir -p /content/drive/MyDrive/tp2-models
    !if [ ! -L models ]; then rm -rf models && ln -s /content/drive/MyDrive/tp2-models models; fi
else:
    # Local: el notebook corre desde notebooks/experiments/, subimos hasta la raíz del repo
    %cd ../..

!nvidia-smi -L || echo "Sin GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU"

## 2. Configuración

In [ ]:
EXPERIMENT = "bidirectional_rnn"
EPOCHS = 15  # candidato final: 10
BIRNN_DROPOUT = 0.3  # candidato final: 0
BIRNN_HEAD_DIM = 128  # candidato final: None (cabeza lineal)
CONFIG_KEYS = ("n_epochs", "class_weight_power", "bidirectional_rnn_weight_decay", "bidirectional_rnn_dropout",
               "bidirectional_rnn_head_dim", "experiments")
# F1 macro en control del candidato final (promedio de las tareas del problema)
BASELINE_CONTROL = {"CapitalizationProblem": 0.6981, "PunctuationProblem": 0.7233}
CLASS_WEIGHT_POWER = 0.25  # igual que el candidato final (src/constants.py: 0.5)
PUNCTUATION_BIRNN_WEIGHT_DECAY = 0.1  # igual que el candidato final (src/problem/punctuation: 1.0)
SIZE = 1_000_000  # máximo de instancias por fuente, igual que las corridas anteriores (~62 mil de train)

# Pisamos los valores antes de importar el pipeline, sin tocar el código del repo.
# run_config los lee de los mismos lugares, así que quedan registrados en la metadata.json de cada modelo.
OVERRIDES = (
    "import runpy, sys; "
    "import src.constants as c; "
    f"c.CLASS_WEIGHT_POWER = {CLASS_WEIGHT_POWER}; "
    "from src.problem import CapitalizationProblem, PunctuationProblem; "
    f"PunctuationProblem.bidirectional_rnn_weight_decay = {PUNCTUATION_BIRNN_WEIGHT_DECAY}; "
    f"CapitalizationProblem.bidirectional_rnn_dropout = PunctuationProblem.bidirectional_rnn_dropout = {BIRNN_DROPOUT}; "
    f"CapitalizationProblem.bidirectional_rnn_head_dim = PunctuationProblem.bidirectional_rnn_head_dim = {BIRNN_HEAD_DIM}; "
    "sys.argv = [\"src.pipeline\"] + sys.argv[1:]; "
    "runpy.run_module(\"src.pipeline\", run_name=\"__main__\")"
)
PIPELINE = f"MPLBACKEND=Agg uv run python -c '{OVERRIDES}'"

def run_reports() -> dict[str, dict]:
    """Reporte de cada corrida, armado con la metadata.json de sus modelos (models/runs/<Problema>/<score>__<modelo>/)."""
    reports = {}
    for path in sorted(Path("models/runs").glob("*/*/metadata.json")):
        meta = json.loads(path.read_text())
        report = reports.setdefault(meta["run"], {"config": meta["config"], "problems": {}})
        report["problems"].setdefault(meta["problem"], {})[meta["model"]] = meta
    return reports


## 3. Entrenamiento

El `.pkl` se guarda en Drive (`MyDrive/tp2-models/<fecha>__<hora>/`) al terminar cada problema.

In [ ]:
before = set(run_reports())
!{PIPELINE} --experiments {EXPERIMENT} --epochs {EPOCHS} \
    --capitalization-size {SIZE} --punctuation-size {SIZE}


reports = run_reports()
# La corrida de este notebook: el otro notebook puede estar escribiendo en la misma carpeta de Drive
new_runs = sorted(run for run in set(reports) - before if EXPERIMENT in (reports[run]["config"]["experiments"] or []))
RUN = new_runs[-1] if new_runs else None
print("Corrida:", RUN)

## 4. Resultados

F1 macro sobre **control** (promedio de las tareas de cada problema) contra el candidato final (`models/reports/precandidato_final`). Si `Δ` es positivo, mejora.

In [ ]:
def task_scores(run: str) -> list[tuple[str, int, dict[str, float]]]:
    """(problema, mejor época, F1 control por tarea) del modelo de este notebook."""
    report = run_reports()[run]
    print(run, "config:", {k: report["config"].get(k) for k in CONFIG_KEYS})
    return [
        (problem, models[EXPERIMENT]["best_epoch"], {t: v["f1_macro"] for t, v in models[EXPERIMENT]["control"].items()})
        for problem, models in report["problems"].items()
        if models.get(EXPERIMENT, {}).get("control")
    ]


if RUN is None:
    print("No se encontró la corrida: completar RUN a mano (campo run de la metadata.json), ej: RUN = '2026-11-18__10-00-00'")
else:
    print(f"{'problema':22} {'época':>5} {'control por tarea':34} {'control':>7} {'candidato final':>15} {'Δ':>7}")
    for problem, epoch, control in task_scores(RUN):
        mean = sum(control.values()) / len(control)
        base = BASELINE_CONTROL[problem]
        tasks = " ".join(f"{t}={v:.3f}" for t, v in control.items())
        print(f"{problem:22} {epoch:5} {tasks:34} {mean:7.3f} {base:15.3f} {mean - base:+7.3f}")